In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive

### Load Model

In [2]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'LINE', 'PREDICTION', 'ODDS','RECOMMENDATION', 'EV%','KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,LINE,PREDICTION,ODDS,RECOMMENDATION,EV%,KELLY_FRACTION,SIGMA FLAG
0,Goga Bitadze,4.5,11.21,105,1,8.67,0.826,Med
1,Steven Adams,4.5,7.77,-125,0,3.58,0.448,Med
2,Marvin Bagley III,5.5,8.80,-140,0,2.73,0.382,Med
3,Tari Eason,10.5,12.74,-118,0,2.44,0.288,Med
4,Tari Eason,10.5,12.74,-123,0,2.21,0.271,Med


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Marvin Bagley III,Steven Adams,7.5,4.5,8.8,7.77,0,0.22,0.108,Med,Med
1,Marvin Bagley III,Tari Eason,7.5,10.5,8.8,12.74,0,0.09,0.043,Med,Med


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Goga Bitadze,Steven Adams,5.0,4.5,11.21,7.77,0,0.83,0.413,Med,Med
1,Goga Bitadze,Tari Eason,5.0,10.5,11.21,12.74,0,0.63,0.315,Med,Med


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'recommendation','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg  
triosPrizepicks = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Goga Bitadze,Tari Eason,Steven Adams,5.0,10.5,4.5,11.21,12.74,7.77,over,over,over,0.869,0.653,0.717,0.3294,0.291,0.075,0.139,0.168,0.98,0.195,0,"(0.3, 22.1)","(1.2, 24.2)","(0.0, 18.7)",21.73,23.0,18.72,5.54,5.87,5.59,Med,Med,Med,Monte Carlo


In [10]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
149,PrizePicks,player_points,Ryan Rollins,Over,13.5,-137,2025-11-01,2025-11-01T21:13:22Z
151,PrizePicks,player_points,Rudy Gobert,Over,11.0,-137,2025-11-01,2025-11-01T21:21:42Z
153,PrizePicks,player_points,Ryan Kalkbrenner,Over,9.5,-137,2025-11-01,2025-11-01T21:21:42Z
155,PrizePicks,player_points,LaMelo Ball,Over,25.5,-137,2025-11-01,2025-11-01T21:21:42Z
157,PrizePicks,player_points,Julius Randle,Over,24.5,-137,2025-11-01,2025-11-01T21:21:42Z
...,...,...,...,...,...,...,...,...
2376,PrizePicks,player_blocks_steals,Bilal Coulibaly,Over,1.5,-137,2025-11-01,2025-11-01T21:22:48Z
2378,PrizePicks,player_blocks_steals,Alperen Sengun,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z
2380,PrizePicks,player_blocks_steals,Josh Minott,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z
2382,PrizePicks,player_blocks_steals,Kevin Durant,Over,1.5,-137,2025-11-02,2025-11-01T21:22:56Z


In [11]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 67 records for player_points to player_points.csv
Saved 48 records for player_rebounds to player_rebounds.csv
Saved 29 records for player_assists to player_assists.csv
Saved 9 records for player_threes to player_threes.csv
Saved 7 records for player_blocks to player_blocks.csv
Saved 13 records for player_steals to player_steals.csv
Saved 35 records for player_field_goals to player_field_goals.csv
Saved 31 records for player_frees_made to player_frees_made.csv
Saved 13 records for player_frees_attempts to player_frees_attempts.csv
Saved 83 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 79 records for player_points_rebounds to player_points_rebounds.csv
Saved 76 records for player_points_assists to player_points_assists.csv
Saved 48 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 20 records for player_turnovers to player_turnovers.csv
Saved 9 records for player_blocks_steals to player_blocks_steals.csv

All category f

In [12]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 66 records for player_points to player_points.csv
Saved 20 records for player_rebounds to player_rebounds.csv
Saved 11 records for player_assists to player_assists.csv
Saved 10 records for player_threes to player_threes.csv
Saved 1 records for player_steals to player_steals.csv
Saved 3 records for player_frees_made to player_frees_made.csv
Saved 76 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 32 records for player_points_rebounds to player_points_rebounds.csv
Saved 26 records for player_points_assists to player_points_assists.csv
Saved 15 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 6 records for player_turnovers to player_turnovers.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
